# PlanetScope Super Resolution — Batch Processing

This notebook demonstrates how to run PlanetScope **Super Resolution** at scale using the **Batch Processing API**. While [PlanetScope_SuperRes.ipynb](PlanetScope_SuperRes.ipynb) covers single-image super resolution via the Process API, this notebook shows how to process large areas efficiently by tiling and parallelizing the work.

PlanetScope imagery at native 3 m/px is first delivered into a data collection with the Subscriptions API. The Batch Processing API then applies the SuperRes model across the entire data collection to produce 2 m/px enhanced imagery stored as Cloud Optimized GeoTIFFs in AWS S3.

**You will learn how to:**
- Define an AOI and create a tiling grid for batch processing
- Run a Super Resolution batch job to produce 2 m enhanced imagery from the 3 m source data
- Monitor the batch job and retrieve results
- Create a BYOC collection and ingest results

## End-to-End Workflow

Running Super Resolution at scale involves five main steps. This notebook covers **Steps 2–4** in detail; the other steps are outlined below so you can set up the full pipeline.

**Step 1 — Subscribe to PlanetScope data with data collection delivery**

Create a [Subscription](https://docs.planet.com/develop/apis/subscriptions/) that delivers PlanetScope imagery into a data collection. The subscription must include a `hosting` block with `"type": "sentinel_hub"` so that the data can be used by Planet's Processing API. Choose an appropriate asset type and optionally enable harmonization (see the *Subscription Setup Details* section below).

**Step 2 — Run Super Resolution batch processing (this notebook)**

Use the Batch Processing API to apply the super resolution model to the 3 m PlanetScope data to produce 2 m enhanced imagery. Results are written to an AWS S3 bucket.

**Step 3 — Create a BYOC collection for super-resolved output (this notebook)**

Create a new, empty [BYOC collection](https://docs.planet.com/develop/apis/byoc/). This collection read the super-resolved imagery from the AWS S3 bucket after ingestion.

**Step 4 — Ingest results into BYOC (this notebook)**

After batch processing completes, ingest the S3 output into the BYOC collection created in Step 3. This makes the super-resolved imagery queryable through the Catalog and Processing APIs, just like any other data collection.

**Step 5 — Create config and visualization layers**

Create a configuration with visualization layers for the super-resolved collection. Example evalscripts are provided at the end of this notebook.

## Documentation

- [Planet Subscriptions API](https://docs.planet.com/develop/apis/subscriptions/)
- [Batch Processing API](https://docs.planet.com/develop/apis/batch-processing/)
- [Bring Your Own COG (BYOC API)](https://docs.planet.com/develop/apis/byoc/)
- [sentinelhub-py SDK](https://sentinelhub-py.readthedocs.io/en/latest/)

## Requirements

Install the required packages before running this notebook:

```bash
pip install sentinelhub geopandas matplotlib contextily boto3
```

You will also need:
- A Planet account with processing units
- OAuth client credentials (`SH_CLIENT_ID` and `SH_CLIENT_SECRET`)
- AWS credentials (`AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`) with read/write access to an S3 bucket for storing the tiling grid and batch output
- A PlanetScope subscription with data delivered to a data collection (see Step 1 above)

## Authentication

This notebook authenticates using OAuth 2.0 client credentials and uses AWS credentials for S3 access.

The following environment variables must be set before running — for example via `export` or your preferred method:

- `SH_CLIENT_ID` and `SH_CLIENT_SECRET` — find these in from your [Planet account](https://insights.planet.com/account/#/)
- `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` — standard AWS credentials with read/write access to your S3 bucket

## Subscription Setup Details

Before running the processing cells, you must have a PlanetScope dataset available in a data collection. The recommended way to set this up is via the [Planet Subscriptions API](https://docs.planet.com/develop/apis/subscriptions/).

---

### Enable data collection delivery in your subscription

To make PlanetScope imagery accessible through the Batch Processing API (and thus eligible for super resolution), your subscription must include a `hosting` block that delivers data directly into a data collction. See the [Subscriptions Delivery — Hosting](https://docs.planet.com/develop/apis/subscriptions/delivery/#hosting) documentation.

```json
"hosting": {
    "type": "sentinel_hub"
}
```

Without this block, imagery is delivered to cloud storage only and cannot be accessed via Batch APIs.

---

### Choose the right asset type for your use case

**For visual or qualitative inspection:** use `ortho_analytic_4b` (Top of Atmosphere reflectance) or `ortho_visual`.

**For downstream analytical processing** — e.g. measuring biophysical properties such as NDVI, LAI, or canopy cover — it is strongly recommended to use **harmonized surface reflectance** assets. Harmonisation adjusts PlanetScope radiometry to match Sentinel-2, ensuring radiometrically consistent cross-sensor comparisons and reliable index calculations.

To enable harmonisation, add the `harmonize` tool to the subscription's `tools` array (see [Subscriptions Tools — Harmonize](https://docs.planet.com/develop/apis/subscriptions/tools/#harmonize)):

```json
"tools": [
    {
        "type": "harmonize",
        "parameters": {
            "target_sensor": "Sentinel-2"
        }
    }
]
```

This applies a per-band radiometric correction so that the surface reflectance values produced by super resolution are consistent with Sentinel-2 Level-2A products and suitable for quantitative analysis.

In [ ]:
import os
import pprint
import re
from collections import defaultdict

#import boto3
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import sentinelhub as sh


# Load credentials from environment variables
SH_CLIENT_ID = os.environ.get("SH_CLIENT_ID")
SH_CLIENT_SECRET = os.environ.get("SH_CLIENT_SECRET")

if not SH_CLIENT_ID or not SH_CLIENT_SECRET:
    raise ValueError(
        "SH_CLIENT_ID and SH_CLIENT_SECRET environment variables must be set. "
        "Find your credentials at https://insights.planet.com/account/#/"
    )

# Load AWS credentials from environment variables
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise ValueError(
        "AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY must be set in your environment variables."
    )

config = sh.SHConfig(
    sh_client_id=SH_CLIENT_ID,
    sh_client_secret=SH_CLIENT_SECRET,
)
config.sh_base_url = "https://services.sentinel-hub.com"
try:
    sh.SentinelHubSession(config=config).token
    print("Successfully authenticated with Sentinel Hub")
except Exception as e:
    raise RuntimeError(f"Sentinel Hub authentication failed: {e}") from e

## Define Area of Interest

The AOI is a ~32 km × 47 km area over La Palma, Canary Islands, Spain. It is stored in `LaPalma.geojson`. We load it with GeoPandas, extract the bounding box, and display it on a map.

In [ ]:
# Load AOI
aoi_gdf = gpd.read_file("LaPalma.geojson")

# Extract bounding box as [minLon, minLat, maxLon, maxLat]
bounds = aoi_gdf.total_bounds  # [minx, miny, maxx, maxy]
bbox_wgs84 = [bounds[0], bounds[1], bounds[2], bounds[3]]

print("AOI bounding box (WGS84):")
print(f"  Min longitude: {bbox_wgs84[0]:.6f}")
print(f"  Min latitude:  {bbox_wgs84[1]:.6f}")
print(f"  Max longitude: {bbox_wgs84[2]:.6f}")
print(f"  Max latitude:  {bbox_wgs84[3]:.6f}")

# Visualize with OSM basemap
aoi_web = aoi_gdf.to_crs(epsg=3857)  # contextily expects Web Mercator

fig, ax = plt.subplots(figsize=(6, 6))
aoi_web.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_title("Area of Interest — La Palma, Canary Islands, Spain", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Configure Batch Parameters

We set the source BYOC collection, time interval, super resolution evalscript, S3 output path, and resolution settings. These parameters are used by the tiling and batch processing steps that follow.

### Choosing the right SuperRes evalscript

Two stored evalscript IDs are available depending on the asset type in your subscription:

| Evalscript ID | Asset type | Subscription tool |
|---|---|---|
| `5a780173-89fc-4f39-9f99-a024f256c713` | `ortho_analytic_4b_sr` — Surface Reflectance (harmonized or not) | Recommended for analytical use |
| `2f84557a-1453-4c5b-bf77-d4892a3fdd78` | `ortho_analytic_4b` — Top of Atmosphere Reflectance (TOAR) | Use when SR is unavailable |

> **Note:** These evalscripts output bands as UINT8. If your workflow requires UINT16 output, contact us for alternative evalscripts.

In [ ]:
# BYOC collection containing scenes to process with Super Resolution
COLLECTION_ID = "<enter-your-byoc-collection-id>"  # Modify with your BYOC collection ID

# Time interval for scene selection
TIME_INTERVAL = (
    "2021-11-08T00:00:00Z",
    "2021-11-08T23:59:59Z",
)  # Change to your desired time range (ISO 8601 format)

# Super Resolution evalscript
# Use the evalscript that matches the asset type in your subscription:
#
# ortho_analytic_4b_sr → Surface Reflectance (harmonized or not) — recommended for analysis
SUPERRES_EVALSCRIPT_ID = "5a780173-89fc-4f39-9f99-a024f256c713"
#
# ortho_analytic_4b    → Top of Atmosphere Reflectance (TOAR)
# SUPERRES_EVALSCRIPT_ID = "2f84557a-1453-4c5b-bf77-d4892a3fdd78"

# S3 Super Resolution output configuration
S3_OUTPUT_URI = (
    "<s3-path-for-super-resolution-output>"  # e.g. "s3://your-bucket/your-prefix"
)
S3_OUTPUT_URI = S3_OUTPUT_URI.rstrip("/")

if not re.match(r"^s3://[a-zA-Z0-9.\-]+/.+", S3_OUTPUT_URI):
    raise ValueError("S3_OUTPUT_URI must be in the format 's3://bucket-name/prefix'")

SUPERRES_PARAMS = {"buffer": 10, "blockSize": 208}

# Resolution settings
INPUT_RESOLUTION = 3  # meters per pixel (PlanetScope pixel size)
SUPER_RES_SCALE_FACTOR = 1.5
OUTPUT_RESOLUTION = INPUT_RESOLUTION / SUPER_RES_SCALE_FACTOR

## Create Tiling Grid and Upload to S3

The Batch Processing API requires a GeoPackage file defining the tile boundaries. We use `UtmGridSplitter` from the `sentinelhub` SDK to split the AOI into fixed-size tiles with a small overlap for edge continuity, then upload the resulting GeoPackage to S3.

In [ ]:
# Build tiling grid with UtmGridSplitter
TILE_SIZE = 1024  # output pixels per tile side
TILE_STRIDE = 1022  # inner pixels (2-pixel overlap for edge continuity)
grid_cell_size = INPUT_RESOLUTION * TILE_STRIDE
overlap = TILE_SIZE - TILE_STRIDE  # 2 pixels overlap

utm_splitter = sh.UtmGridSplitter(
    aoi_gdf.geometry.values.tolist(),
    crs=sh.CRS.WGS84,
    bbox_size=(grid_cell_size, grid_cell_size),
)
bbox_list = utm_splitter.get_bbox_list(buffer=overlap / TILE_STRIDE)

# Group by UTM CRS (handles AOIs that span multiple UTM zones)
crs_to_bboxes = defaultdict(list)
for bbox in bbox_list:
    crs_to_bboxes[bbox.crs].append(bbox)

tile_gdfs = []
for crs, bboxes in crs_to_bboxes.items():
    epsg_str = crs.ogc_string()
    gdf = gpd.GeoDataFrame(
        geometry=[bbox.geometry for bbox in bboxes],
        crs=epsg_str,
    )
    gdf["identifier"] = gdf["geometry"].map(
        lambda geom, crs=epsg_str: (
            f"tile-{str(geom.bounds[0]).replace('.', '_')}"
            f"-{str(geom.bounds[3]).replace('.', '_')}"
            f"-{crs.replace(':', '_')}"
        )
    )
    gdf["width"] = TILE_SIZE
    gdf["height"] = TILE_SIZE
    gdf["resolution"] = OUTPUT_RESOLUTION
    tile_gdfs.append(gdf[["identifier", "width", "height", "resolution", "geometry"]])

total_tiles = sum(len(gdf) for gdf in tile_gdfs)
print(
    f"Created {total_tiles} tiles ({TILE_SIZE}×{TILE_SIZE} px at {OUTPUT_RESOLUTION} m/px) "
    f"across {len(tile_gdfs)} UTM zone(s)"
)
for gdf in tile_gdfs:
    print(f"  {gdf.crs}: {len(gdf)} tiles")

all_ids = pd.concat([gdf["identifier"] for gdf in tile_gdfs])
assert all_ids.is_unique, "Tile identifiers are not unique!"

# Save as GeoPackage — one layer per CRS to keep tiles axis-aligned
gpkg_path = "grid.gpkg"
if os.path.exists(gpkg_path):
    os.remove(gpkg_path)
for gdf in tile_gdfs:
    epsg = gdf.crs.to_epsg()
    gdf.to_file(gpkg_path, driver="GPKG", layer=str(epsg))

# Upload to S3
bucket, prefix = S3_OUTPUT_URI.removeprefix("s3://").split("/", 1)
grid_key = f"{prefix}/grid.gpkg"
S3_GRID_URL = f"s3://{bucket}/{grid_key}"

s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)
s3_client.upload_file(gpkg_path, bucket, grid_key)
print(f"Uploaded tiling grid to: {S3_GRID_URL}")

In [ ]:
# Visualize tiling grid
fig, ax = plt.subplots(figsize=(6, 6))
for gdf in tile_gdfs:
    gdf.to_crs(epsg=3857).plot(ax=ax, facecolor="none", edgecolor="red", linewidth=1)
aoi_web.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=2)
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_title("Tiling Grid (red) over AOI (black)", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Run SuperRes Batch Job

With the 3 m/px PlanetScope data delivered into the BYOC collection via your subscription, we can now apply SuperRes to produce 2 m/px enhanced imagery. Instead of an inline evalscript, we reference a **stored evalscript ID** that invokes the AI model. Additional model parameters (`buffer` and `blockSize`) are passed via `evalscriptParams`.

The output tiles contain 6 bands following PlanetScope source band ordering:
- Band 1: Blue (super-resolved)
- Band 2: Green (super-resolved)
- Band 3: Red (super-resolved)
- Band 4: Other
- Band 5: Confidence score (0–100)
- Band 6: Alpha/DataMask

In [ ]:
# Batch Process Client
batch_client = sh.BatchProcessClient(config=config)

# GeoPackage input — references the grid uploaded in the previous step
s3_grid_spec = sh.BatchProcessClient.s3_specification(
    url=S3_GRID_URL,
    access_key=AWS_ACCESS_KEY_ID,
    secret_access_key=AWS_SECRET_ACCESS_KEY,
)

# Raster output to S3
s3_output_spec = sh.BatchProcessClient.s3_specification(
    url=S3_OUTPUT_URI,
    access_key=AWS_ACCESS_KEY_ID,
    secret_access_key=AWS_SECRET_ACCESS_KEY,
)

# Build the Super Resolution Batch API request
superres_request = sh.SentinelHubRequest(
    evalscript=SUPERRES_EVALSCRIPT_ID,
    input_data=[
        sh.SentinelHubRequest.input_data(
            data_collection=sh.DataCollection.define_byoc(COLLECTION_ID),
            time_interval=TIME_INTERVAL,
            mosaicking_order=sh.MosaickingOrder.LEAST_RECENT,
            upsampling=sh.ResamplingType.BILINEAR,
            downsampling=sh.ResamplingType.BILINEAR,
        )
    ],
    responses=[sh.SentinelHubRequest.output_response("default", sh.MimeType.TIFF)],
    bbox=sh.BBox(bbox=bbox_wgs84, crs=sh.CRS("EPSG:4326")),
    config=config,
)
superres_request.payload["evalscriptParams"] = SUPERRES_PARAMS

batch_input = sh.BatchProcessClient.geopackage_input(s3_grid_spec)
batch_output = sh.BatchProcessClient.raster_output(
    delivery=s3_output_spec,
    cog_output=True,
)

print("Batch API payload:")
pprint.pprint(superres_request.payload)

# Create
print("\nCreating Super Resolution batch request...")
batch_request = batch_client.create(
    process_request=superres_request,
    input=batch_input,
    output=batch_output,
)
print(f"Request ID : {batch_request.request_id}")
print(f"Status     : {batch_request.status.value}")

# Start
print("\nStarting Super Resolution batch job...")
batch_client.start_job(batch_request)

# Monitor (analysis → processing → done)
print("Monitoring batch job (this may take several minutes)...")
batch_request = sh.monitor_batch_process_job(
    request=batch_request,
    client=batch_client,
    sleep_time=60,
)

# Results
if batch_request.status == sh.BatchRequestStatus.DONE:
    print("\nSuper Resolution processing complete.")
    print(f"Results  : {S3_OUTPUT_URI}/")
    print(f"Request ID: {batch_request.request_id}")
else:
    print(f"\nBatch ended with status: {batch_request.status.value}")

## Ingest Results into BYOC

After batch processing completes, ingest the S3 output tiles into a BYOC collection so the super-resolved imagery is queryable via the Catalog and Processing APIs.

For the full ingestion workflow using `sentinelhub-py`, see the [BYOC request examples](https://sentinelhub-py.readthedocs.io/en/latest/examples/byoc_request.html#Add-multiple-tiles-to-a-single-collection).

The super-resolution output encodes all bands in a single TIFF file. When configuring the BYOC collection, define the band mapping so that evalscripts can reference bands by name (e.g. `sample.blue`, `sample.confidence`):

In [ ]:
byoc_additional_data = sh.ByocCollectionAdditionalData(
    bands={
        "blue": sh.ByocCollectionBand(
            source="default", band_index=1, bit_depth=8, sample_format="UINT"
        ),
        "green": sh.ByocCollectionBand(
            source="default", band_index=2, bit_depth=8, sample_format="UINT"
        ),
        "red": sh.ByocCollectionBand(
            source="default", band_index=3, bit_depth=8, sample_format="UINT"
        ),
        "other": sh.ByocCollectionBand(
            source="default", band_index=4, bit_depth=8, sample_format="UINT"
        ),
        "confidence": sh.ByocCollectionBand(
            source="default", band_index=5, bit_depth=8, sample_format="UINT", no_data=0
        ),
        "alpha": sh.ByocCollectionBand(
            source="default", band_index=6, bit_depth=8, sample_format="UINT"
        ),
    }
)

byoc_superres_collection = sh.ByocCollection(
    "Super resolution output collection",
    additional_data=byoc_additional_data,
    s3_bucket=bucket,
)
byoc = sh.SentinelHubBYOC(config=config)
byoc_superres_collection = byoc.create_collection(byoc_superres_collection)
print(f"Created BYOC collection: {byoc_superres_collection['id']}")

In [ ]:
# Ingest tiles into BYOC collection (skips existing tiles)
total_ingested = 0
total_skipped = 0

for gdf in tile_gdfs:
    for tile_id in gdf["identifier"].values:
        tile_path = f"{prefix}/{batch_request.request_id}/{tile_id}/(BAND).tif"
        try:
            byoc.create_tile(
                byoc_superres_collection,
                {"path": tile_path, "sensingTime": TIME_INTERVAL[0]},
            )
            total_ingested += 1
        except sh.DownloadFailedException as e:
            error_data = e.request_exception.response.json()
            error_code = error_data.get("error", {}).get("code", "")
            if (
                e.request_exception.response.status_code == 409
                and error_code == "COMMON_UNIQUE_KEY_VIOLATION"
            ):
                total_skipped += 1
            else:
                raise

print(f"Ingested {total_ingested} tiles, skipped {total_skipped} existing")

## Summary

In this notebook we:

1. Defined an AOI over La Palma and created a tiling grid for batch processing
2. Uploaded the grid to S3 as a GeoPackage for the Batch API to consume
3. Ran a **Super Resolution batch job** to enhance the 3 m/px PlanetScope source data to 2 m/px
4. Monitored the job to completion and stored results in S3
5. Created a BYOC collection for super resolution data and ingested data into it.

### Next steps

- **Visualize results:** Once ingested, create a new [configuration](https://insights.planet.com/analyze/configurations/) with a visualization layer (see the two examples below), then [browse the super-resolved imagery](https://insights.planet.com/analyze/browser/).
- **Try your own AOI:** Replace `LaPalma.geojson` with a different bounding polygon

---

### Example evalscript — True Color

```javascript
//VERSION=3
//True Color

function setup() {
    return {
        input: [
            "blue",
            "green",
            "red",
            "alpha"
        ],
        output: {
            bands: 4
        }
    };
}

function evaluatePixel(sample) {
    return [sample.red / 100, sample.green / 100, sample.blue / 100, sample.alpha/255
    ];
}
```

### Example evalscript — Confidence

```javascript
//VERSION=3
//Confidence

function setup() {
    return {
        input: [
            "confidence",
            "alpha"
        ],
        output: {
            bands: 4
        }
    };
}

const viz = new ColorRampVisualizer([
    [0,   [0.647, 0.000, 0.149]],
    [10,  [0.843, 0.188, 0.153]],
    [20,  [0.957, 0.427, 0.263]],
    [30,  [0.992, 0.682, 0.380]],
    [40,  [0.996, 0.878, 0.545]],
    [50,  [1.000, 1.000, 0.749]],
    [60,  [0.851, 0.937, 0.545]],
    [70,  [0.651, 0.851, 0.416]],
    [80,  [0.400, 0.741, 0.388]],
    [90,  [0.102, 0.596, 0.314]],
    [100, [0.000, 0.408, 0.216]],
]);

function evaluatePixel(sample) {
    const rgb = viz.process(sample.confidence);
    return [...rgb, sample.alpha/255];
}
```